# 🎮 Project Prometheus: Interactive Game Playing Demo

## Three Games with Increasing Complexity

This notebook demonstrates:
- **Connect 4** (Simple) - Get 4 in a row
- **Othello** (Intermediate) - Flip opponent pieces
- **Draughts** (Complex) - Captures and kings

All with:
- 🎨 **Full color visualization**
- 🧠 **Agent thinking display**
- 📊 **Strategy analysis**
- 🔄 **Move-by-move replay**

In [ ]:
# Setup
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output
import time

# Configure matplotlib for notebook
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100

# Import Prometheus modules
sys.path.insert(0, os.path.abspath('.'))

from prometheus.game_suite import Connect4, Othello
from benchmarks.prometheus_bench_v0_2 import DraughtsGame
from prometheus.notebook_viz import GameBoardViz, AgentThinkingViz
from demo_game_visualization import HeuristicAgent

print("✓ All modules loaded successfully!")
print("\n🎮 Ready to play!")

---

## 🟢 Demo 1: Connect 4 (Simple Game)

The simplest game - drop tokens to get 4 in a row.

**Agent Strategy:**
1. Win if possible
2. Block opponent wins
3. Prefer center columns
4. Random from remaining

In [ ]:
# Create Connect4 game
game = Connect4()
agent1 = HeuristicAgent("Red Agent", "connect4")
agent2 = HeuristicAgent("Blue Agent", "connect4")

print("🎮 Connect 4 Game Started!")
print(f"   {agent1.name} (●) vs {agent2.name} (○)\n")

# Show initial board
fig = GameBoardViz.visualize_connect4(game.board, title="Connect 4 - Start")
plt.show()

print("\n✓ Game initialized. Run next cell to play moves.")

In [ ]:
# Play a few moves with visualization
agents = {1: agent1, -1: agent2}
move_count = 0
max_moves = 10  # Show first 10 moves

while game.result.value == 'ongoing' and move_count < max_moves:
    current_agent = agents[game.current_player]
    player_name = agent1.name if game.current_player == 1 else agent2.name
    symbol = "●" if game.current_player == 1 else "○"
    
    print(f"\n{'='*70}")
    print(f"Move {move_count + 1}: {player_name} ({symbol})")
    print(f"{'='*70}")
    
    # Agent thinks
    move = current_agent.select_move(game.get_state())
    
    # Show thinking
    if current_agent.thinking:
        fig = AgentThinkingViz.show_thinking(
            player_name,
            current_agent.thinking,
            f"Column {move}"
        )
        plt.show()
    
    # Make move
    if move is not None:
        game.make_move(move)
        move_count += 1
        
        # Visualize board
        fig = GameBoardViz.visualize_connect4(
            game.board,
            last_move=game.last_move,
            title=f"Connect 4 - After Move {move_count}"
        )
        plt.show()
    else:
        break
    
    # Check for win
    if game.result.value != 'ongoing':
        print("\n" + "="*70)
        if game.winner == 1:
            print(f"🏆 {agent1.name} WINS!")
        elif game.winner == -1:
            print(f"🏆 {agent2.name} WINS!")
        else:
            print("🤝 DRAW!")
        print("="*70)
        break

if move_count >= max_moves:
    print("\n⏸️ Showing first 10 moves only. Continue in next cell or try other games!")

---

## 🟡 Demo 2: Othello (Intermediate Game)

More complex - flip opponent pieces by flanking them.

**Agent Strategy:**
1. Take corners (strongest positions)
2. Avoid squares next to corners
3. Maximize flipped pieces
4. Control center

In [ ]:
# Create Othello game
game = Othello()
agent1 = HeuristicAgent("Black Agent", "othello")
agent2 = HeuristicAgent("White Agent", "othello")

print("🎮 Othello Game Started!")
print(f"   {agent1.name} (●) vs {agent2.name} (○)\n")

# Show initial board with valid moves
valid_moves = game.get_valid_moves()
fig = GameBoardViz.visualize_othello(
    game.board,
    valid_moves=valid_moves,
    title="Othello - Start (purple = valid moves)"
)
plt.show()

print(f"\n✓ Game initialized. {len(valid_moves)} valid opening moves.")

In [ ]:
# Play several moves with visualization
agents = {1: agent1, -1: agent2}
move_count = 0
max_moves = 8  # Show first 8 moves

while game.result.value == 'ongoing' and move_count < max_moves:
    current_agent = agents[game.current_player]
    player_name = agent1.name if game.current_player == 1 else agent2.name
    symbol = "●" if game.current_player == 1 else "○"
    
    print(f"\n{'='*70}")
    print(f"Move {move_count + 1}: {player_name} ({symbol})")
    print(f"{'='*70}")
    
    # Get valid moves
    valid_moves = game.get_valid_moves()
    
    if not valid_moves:
        print("No valid moves - passing turn")
        game.make_move(None)
        continue
    
    # Agent thinks
    move = current_agent.select_move(game.get_state())
    
    # Show thinking
    if current_agent.thinking:
        fig = AgentThinkingViz.show_thinking(
            player_name,
            current_agent.thinking,
            f"Position {move}"
        )
        plt.show()
    
    # Make move
    if move in valid_moves:
        game.make_move(move)
        move_count += 1
        
        # Visualize board
        next_valid = game.get_valid_moves() if game.result.value == 'ongoing' else []
        fig = GameBoardViz.visualize_othello(
            game.board,
            valid_moves=next_valid,
            last_move=move,
            title=f"Othello - After Move {move_count}"
        )
        plt.show()
    else:
        break
    
    # Check for end
    if game.result.value != 'ongoing':
        print("\n" + "="*70)
        if game.winner == 1:
            print(f"🏆 {agent1.name} WINS!")
        elif game.winner == -1:
            print(f"🏆 {agent2.name} WINS!")
        else:
            print("🤝 DRAW!")
        print("="*70)
        break

if move_count >= max_moves:
    print("\n⏸️ Showing first 8 moves only. Game continues...")

---

## 🔴 Demo 3: Draughts (Complex Game)

Most complex - captures, forced moves, kings, and multi-jumps.

**Agent Strategy:**
1. Prefer captures (mandatory in real draughts)
2. Advance pieces toward opponent side
3. Protect back rank
4. Pursue kingship

In [ ]:
# Create Draughts game
game = DraughtsGame()
agent1 = HeuristicAgent("Red Agent", "draughts")
agent2 = HeuristicAgent("White Agent", "draughts")

print("🎮 Draughts Game Started!")
print(f"   {agent1.name} (●) vs {agent2.name} (○)\n")

# Show initial board
fig = GameBoardViz.visualize_draughts(
    game.board,
    title="Draughts - Start"
)
plt.show()

print("\n✓ Game initialized. Each player has 12 pieces.")

In [ ]:
# Play several moves with visualization
agents = {1: agent1, -1: agent2}
move_count = 0
max_moves = 10  # Show first 10 moves

while not game.game_over and move_count < max_moves:
    current_agent = agents[game.current_player]
    player_name = agent1.name if game.current_player == 1 else agent2.name
    symbol = "●" if game.current_player == 1 else "○"
    
    print(f"\n{'='*70}")
    print(f"Move {move_count + 1}: {player_name} ({symbol})")
    print(f"{'='*70}")
    
    # Get valid moves
    valid_moves = game.get_valid_moves(game.current_player)
    
    if not valid_moves:
        print("No valid moves - game over")
        break
    
    # Agent thinks
    move = current_agent.select_move(game.get_state())
    
    # Show thinking
    if current_agent.thinking:
        fig = AgentThinkingViz.show_thinking(
            player_name,
            current_agent.thinking[:3],  # Show top 3 thoughts
            f"Move from {move[0]} to {move[1]}" if move else "No move"
        )
        plt.show()
    
    # Make move
    if move in valid_moves:
        game.make_move(move)
        move_count += 1
        
        # Visualize board
        next_valid = game.get_valid_moves(game.current_player) if not game.game_over else []
        fig = GameBoardViz.visualize_draughts(
            game.board,
            valid_moves=next_valid,
            last_move=move,
            title=f"Draughts - After Move {move_count}"
        )
        plt.show()
    else:
        break
    
    # Check for end
    if game.game_over:
        print("\n" + "="*70)
        if game.winner == 1:
            print(f"🏆 {agent1.name} WINS!")
        elif game.winner == -1:
            print(f"🏆 {agent2.name} WINS!")
        else:
            print("🤝 DRAW!")
        print("="*70)
        break

if move_count >= max_moves:
    print("\n⏸️ Showing first 10 moves only. Game continues...")

---

## 📊 Summary & Next Steps

You've seen three games with increasing complexity:

| Game | Complexity | Board Size | Key Features |
|------|-----------|------------|-------------|
| Connect 4 | Simple | 7×6 | Drop tokens, 4-in-a-row |
| Othello | Intermediate | 8×8 | Flip pieces, corners crucial |
| Draughts | Complex | 8×8 | Captures, kings, multi-jumps |

### What's Next?

1. **Evolution Demo** - See agents actually EVOLVE and improve over generations
2. **Brain Map** - Visualize agent architecture and thinking patterns
3. **Fitness Landscape** - Watch how agent strategies adapt
4. **Transfer Learning** - See patterns learned in Connect4 applied to Draughts

### Try It Yourself!

Modify the heuristics in `demo_game_visualization.py` to create your own agent strategies!

---

**Project Prometheus v0.69+**  
*Recursive Self-Improvement through Game Playing*  
*Running on Jetson Orin Nano with Local GPU Inference*